# 04 - Modeling and Evaluation
Covers Deliverable D


In [ ]:
import os, sys
import pandas as pd
import numpy as np

def resolve_paths():
    for p in ['../data/raw', 'data/raw', 'team_6/data/raw', '../../data/raw']:
        if os.path.exists(os.path.join(p, 'crop_yield_train.csv')):
            raw = p
            base = os.path.dirname(p)
            proc = os.path.join(base, 'processed')
            fig = os.path.join(os.path.dirname(base), 'figures')
            mod = os.path.join(os.path.dirname(base), 'models')
            return raw, proc, fig, mod
    return '../data/raw', '../data/processed', '../figures', '../models'

RAW_DIR, PROCESSED_DIR, FIGURES_DIR, MODELS_DIR = resolve_paths()
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib

train = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_train.csv'))
X = train.drop(columns=['plot_id', 'yield_tons_per_ha'])
y = train['yield_tons_per_ha']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

print("Model pipeline constructed successfully. Master data ready!")

